# H\'enon

## Generate data

In [ ]:
import numpy as np
import os

N_DIM = 10
N_STEPS = 5000

def generateHenon(n_steps=10000, n_dim=10):
    X = np.random.rand(n_steps + 1, n_dim)
    a = 1.4
    b = 0.3
    for i in range(n_steps):
        x_next = np.zeros_like(X[i])
        for j in range(len(x_next)):
            if j != 0:
                x_next[j] = a - (b * X[i][j-1] + (1 - b) * X[i][j])**2 + b * X[i-1][j]
            else:
                x_next[j] = a - X[i][j] ** 2 + b * X[i-1][j]
        X[i+1] = x_next
    return X

X = generateHenon(n_steps=N_STEPS, n_dim=N_DIM)

# Check if any element has become Infinity or very large
print(np.min(X), np.max(X))

## Save the full data

In [ ]:
# NAME should be of the form "<system>_<n_steps>_<n_dim>.npy"
# This will be utilized later to load the data and extract system, n_steps, n_dim from the filename
filename = f"henon_{N_STEPS}_{N_DIM}.npy"

parent = os.path.abspath('')
os.makedirs(os.path.join(parent, 'data', 'henon'), exist_ok=True)

np.save(os.path.join(parent, 'data', 'henon', filename), X)

## Normalization/Standardization, Chronological chunking, & splitting

In [ ]:
import os
from utils_data import preprocess_all

parent = os.path.abspath('')
artifact = f"henon_{N_STEPS}_{N_DIM}.npy"

CONTEXT = 50
VAL_FRAC = 0.15
TEST_FRAC = 0.15

written = preprocess_all(os.path.join(parent, 'data', 'henon'), artifact, context=CONTEXT, val_frac=VAL_FRAC, test_frac=TEST_FRAC)
print("\nWrote:")
for w in written:
    print(" ", w)

# Lorenz-96

## Generate data

In [ ]:
from scipy.integrate import odeint
import os, numpy as np

N_DIM = 10
N_STEPS = 5000

def make_var_stationary(beta, radius=0.97):
    '''Rescale coefficients of VAR model to make stable.'''
    p = beta.shape[0]
    lag = beta.shape[1] // p
    bottom = np.hstack((np.eye(p * (lag - 1)), np.zeros((p * (lag - 1), p))))
    beta_tilde = np.vstack((beta, bottom))
    eigvals = np.linalg.eigvals(beta_tilde)
    max_eig = max(np.abs(eigvals))
    nonstationary = max_eig > radius
    if nonstationary:
        return make_var_stationary(0.95 * beta, radius)
    else:
        return beta

def simulate_var(p, T, lag, sparsity=0.2, beta_value=1.0, sd=0.1, seed=0):
    if seed is not None:
        np.random.seed(seed)

    # Set up coefficients and Granger causality ground truth.
    GC = np.eye(p, dtype=int)
    beta = np.eye(p) * beta_value

    num_nonzero = int(p * sparsity) - 1
    for i in range(p):
        choice = np.random.choice(p - 1, size=num_nonzero, replace=False)
        choice[choice >= i] += 1
        beta[i, choice] = beta_value
        GC[i, choice] = 1

    beta = np.hstack([beta for _ in range(lag)])
    beta = make_var_stationary(beta)

    # Generate data.
    burn_in = 100
    errors = np.random.normal(scale=sd, size=(p, T + burn_in))
    X = np.zeros((p, T + burn_in))
    X[:, :lag] = errors[:, :lag]
    for t in range(lag, T + burn_in):
        X[:, t] = np.dot(beta, X[:, (t-lag):t].flatten(order='F'))
        X[:, t] += + errors[:, t-1]

    return X.T[burn_in:], beta, GC

def lorenz(x, t, F):
    '''Partial derivatives for Lorenz-96 ODE.'''
    p = len(x)
    dxdt = np.zeros(p)
    for i in range(p):
        dxdt[i] = (x[(i+1) % p] - x[(i-2) % p]) * x[(i-1) % p] - x[i] + F
    return dxdt

def generateLorenz96(n_dim, T, F=10.0, delta_t=0.1, sd=0.1, burn_in=1000, seed=42):
    if seed is not None:
        np.random.seed(seed)

    parent = os.path.abspath('')

    # Use scipy to solve ODE.
    x0 = np.random.normal(scale=0.01, size=n_dim)
    t = np.linspace(0, (T + burn_in) * delta_t, T + burn_in)
    X = odeint(lorenz, x0, t, args=(F,))
    X += np.random.normal(scale=sd, size=(T + burn_in, n_dim))

    return X[burn_in:]

X = generateLorenz96(n_dim=N_DIM, T=N_STEPS)

# Check if any element has become Infinity or very large
print(np.min(X), np.max(X))

## Save the full data

In [ ]:
# NAME should be of the form "<system>_<n_steps>_<n_dim>.npy"
# This will be utilized later to load the data and extract system, n_steps, n_dim from the filename
filename = f"lorenz_{N_STEPS}_{N_DIM}.npy"

parent = os.path.abspath('')
os.makedirs(os.path.join(parent, 'data', 'lorenz'), exist_ok=True)
np.save(os.path.join(parent, 'data', 'lorenz', filename), X)

## Normalization/Standardization, Chronological chunking, & splitting

In [ ]:
import os
from utils_data import preprocess_all

parent = os.path.abspath('')
artifact = f"lorenz_{N_STEPS}_{N_DIM}.npy"

CONTEXT = 50
VAL_FRAC = 0.15
TEST_FRAC = 0.15

written = preprocess_all(os.path.join(parent, 'data', 'lorenz'), artifact, context=CONTEXT, val_frac=VAL_FRAC, test_frac=TEST_FRAC)
print("\nWrote:")
for w in written:
    print(" ", w)

# OBD-II

Requires manual preprocessing in some parts and careful handling

## Filter out non-OBD columns and save a separate `.csv`

In [ ]:
import pandas as pd
import os

parent = os.path.abspath('')
dataset = os.path.join(parent, 'data', 'VehicularData(anonymized).csv')

df1 = pd.read_csv(dataset)
print(f"Columns : {df1.columns}")

###################################################
############ DROP NON-OBD COLUMNS #################
###################################################

drop_cols = ['GPS_Time', 'Device_Time', 'Trip', 'GPS_Long', 'GPS_Lat', 'GPS_Speed_Ms', 'GPS_HDOP', 'GPS_Bearing', 'Gx', 'Gy', 'Gz', 'G_Calibrated', 'OBD_Trip_KPL_Average',
             'Device_Barometer_M', 'GPS_Altitude_M', 'GPS_Accuracy_M', 'GPS_Speed_Km', 'Device_Trip_Dist_Km', 'OBD_Adapter_Voltage', 'Device_Fuel_Remaining',
             'Device_Cost_Km_Inst', 'Device_Cost_Km_Trip', 'Context', 'Reaction_Time', 'Speed_RPM_Relation', 'KPL_Instant']

for col in drop_cols:
    try:
        df1.drop(col, axis=1, inplace=True)
    except:
        pass
columns = df1.columns
print(f"Columns after dropping :\n{columns}\nCount : {len(columns)}")
print(f"Number of samples : {len(df1)}")

os.makedirs(os.path.join(parent, 'data', 'obd2'), exist_ok=True)
df1.to_csv(os.path.join(parent, 'data', 'obd2', 'vehicular_modified.csv'), index=False)

## Load the filtered `.csv`, separate vehicles, and inspect structure

In [ ]:
import numpy as np
import pandas as pd
import os, json, pickle

parent = os.path.abspath('')
csv_path = os.path.join(parent, 'data', 'obd2', 'vehicular_modified.csv')

# ----- parameters you may change -----
CONTEXT   = 30
VAL_FRAC  = 0.10
TEST_FRAC = 0.20
ID_COLS   = ['Car_Id', 'Person_Id']
CO2_TARGETS = ['OBD_CO2_gkm_Average', 'OBD_CO2_gkm_Instant']  # recorded, not special
OUT_DIR   = os.path.join(parent, 'data', 'obd2', f'obd2_ctx{CONTEXT}')
os.makedirs(OUT_DIR, exist_ok=True)
print('context', CONTEXT, '| val', VAL_FRAC, '| test', TEST_FRAC, '| out', OUT_DIR)

df = pd.read_csv(csv_path)
feature_cols = [c for c in df.columns if c not in ID_COLS]
co2_idx = [feature_cols.index(c) for c in CO2_TARGETS]
print(f'Total rows: {len(df)} | features: {len(feature_cols)}')
print('Feature columns:', feature_cols)
print('CO2 target indices in feature space:', dict(zip(CO2_TARGETS, co2_idx)))

df1 = df[df['Car_Id'] == 1].copy()      # in-distribution vehicle
df2 = df[df['Car_Id'] == 2].copy()      # cross-vehicle OOD
print(f'\nVehicle 1: {len(df1)} rows | Vehicle 2: {len(df2)} rows')
print('\nVehicle 1 per-driver rows (file order):')
print(df1.groupby('Person_Id', sort=False).size().to_string())
print('\nVehicle 2 per-driver rows (file order):')
print(df2.groupby('Person_Id', sort=False).size().to_string())

## Build raw per-driver segments (no scaling yet)

In [ ]:
from utils_data import split_series

# Vehicle 1: split each driver's timeline, keep segments grouped by driver
v1_segments = {'train': [], 'val': [], 'test': []}
for pid, sub in df1.groupby('Person_Id', sort=False):
    arr = sub[feature_cols].to_numpy(np.float64)
    tr, va, te = split_series(arr, VAL_FRAC, TEST_FRAC) # Split driver timewise
    v1_segments['train'].append((pid, tr))
    v1_segments['val'].append((pid, va))
    v1_segments['test'].append((pid, te))

# Vehicle 2: each driver block kept whole; all pooled into the OOD set
v2_segments = [(pid, sub[feature_cols].to_numpy(np.float64)) for pid, sub in df2.groupby('Person_Id', sort=False)]

# report rows per split and warn about segments too short to window
for split in ['train', 'val', 'test']:
    total = sum(len(s) for _, s in v1_segments[split])
    short = [pid for pid, s in v1_segments[split] if len(s) < CONTEXT]
    print(f"V1 {split:5s}: {total:7d} rows over {len(v1_segments[split])} drivers"
          f"{'  | too short for a window: drivers ' + str(short) if short else ''}")
print(f"V2 ood  : {sum(len(s) for _, s in v2_segments):7d} rows over {len(v2_segments)} drivers")

## Fit scaler on train only, chunk, and write the three npz files

In [ ]:
from utils_data import make_obd_variant

for kind in ['raw', 'minmax', 'std']:
    arrays, scaler = make_obd_variant(kind, v1_segments, v2_segments, CONTEXT)
    meta = {
        'dataset': 'obd', 'scaling': kind, 'context': CONTEXT,
        'val_frac': VAL_FRAC, 'test_frac': TEST_FRAC,
        'split': 'per_driver_temporal (B1)', 'scaler_fit_on': 'vehicle1_train',
        'in_distribution_vehicle': 1, 'ood_vehicle': 2,
        'feature_names': feature_cols, 'co2_target_idx': co2_idx,
        'window_counts': {k: int(v.shape[0]) for k, v in arrays.items() if k.endswith('left')},
    }
    fname = os.path.join(OUT_DIR, f'obd_{kind}_ctx{CONTEXT}.npz')
    np.savez_compressed(fname, feature_names=np.array(feature_cols), meta=np.array(json.dumps(meta)), **arrays)
    if scaler is not None:
        with open(os.path.join(OUT_DIR, f'scaler_{kind}.pkl'), 'wb') as f:
            pickle.dump(scaler, f)
    print(f"[{kind:6s}] " + "  ".join(
        f"{k.replace('X_','').replace('_left','')}={v.shape[0]}"
        for k, v in arrays.items() if k.endswith('left')) + f"  -> {os.path.basename(fname)}")

print('\nDone. Files in:', OUT_DIR)

## Verify a saved file

In [ ]:
from utils_data import load_preprocessed

splits, feats, meta = load_preprocessed(os.path.join(OUT_DIR, 'obd_minmax_ctx30.npz'))
print('scaling:', meta['scaling'], '| split:', meta['split'])
print('shapes:', {k: v[0].shape for k, v in splits.items()})
tr = splits['train'][0].reshape(-1, len(feats))
te = splits['test'][0].reshape(-1, len(feats))
print('minmax train min/max:', round(float(tr.min()), 3), round(float(tr.max()), 3))
print('test  min/max (may exceed [0,1]):', round(float(te.min()), 3), round(float(te.max()), 3))